# Droplet platform main UI

This notebook is intentionally thin. Acquisition, detection, tracking, droplet-wise photometry, video replay and recording are implemented in `src/droplet_platform`.

Typical workflow:
1. Choose `Camera` or `Video file` source.
2. Tune detector parameters while watching the preview.
3. Save snapshots or raw/overlay videos if needed.
4. Record droplet-wise intensity traces into a TSV file.


In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src = repo_root / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from droplet_platform.ui import AppState, make_basic_controls, display_controls, bind_basic_callbacks
from droplet_platform.detection import DropletDetectorCuPy, make_neural_detector
from droplet_platform.utils import load_yaml

print(f"Repository root: {repo_root}")


C:\Users\stepo\.ReNew_python\Lib\site-packages\cupyx\jit\_interface.py:173: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


Repository root: C:\Users\stepo\Desktop\Presentation\Tracker_Dev\droplet_platform_repo_public_ui_debug


In [2]:
cfg_path = repo_root / 'configs' / 'default.yaml'
cfg = load_yaml(cfg_path) if cfg_path.exists() else {}
cfg


{'camera': {'fps': 20.0, 'exposure_ms': 50.0, 'timeout_ms': 5000},
 'source': {'kind': 'camera',
  'video_path': '',
  'video_loop': True,
  'fake_realtime': True},
 'detector': {'backend': 'correlation',
  'percentile': 89.0,
  'downscale_factor': 2,
  'droplet_radius_px': 100,
  'min_distance': 40,
  'corr_backend': 'fft',
  'cpu_downscale': True,
  'peak_downscale': 1,
  'model_path': 'models/droplet_net.pt',
  'neural_threshold': 0.35},
 'tracker': {'max_dist_px': 400.0,
  'max_age': 20,
  'vel_alpha': 0.35,
  'score_alpha': 0.2},
 'preview': {'colorize': True,
  'draw_ids': True,
  'draw_detections': False,
  'beep_on_crossing': True,
  'reset_tracker_on_start': True},
 'recording': {'root_dir': 'experiments',
  'duration_s': 60.0,
  'roi_radius_px': 80,
  'output_name': 'droplet_intensity.tsv',
  'raw_video_name': 'raw_video.avi',
  'overlay_video_name': 'overlay_video.avi'}}

In [3]:
det_cfg = cfg.get('detector', {})
backend = det_cfg.get('backend', 'correlation')

if backend == 'neural':
    detector = make_neural_detector(
        repo_root / det_cfg.get('model_path', 'models/droplet_net.pt'),
        downscale_factor=det_cfg.get('downscale_factor', 2),
        threshold=det_cfg.get('neural_threshold', 0.35),
        min_distance=det_cfg.get('min_distance', 40),
    )
else:
    if DropletDetectorCuPy is None:
        raise RuntimeError('DropletDetectorCuPy is unavailable. Install the matching cupy-cudaXX package or switch detector.backend to neural.')
    detector = DropletDetectorCuPy(
        percentile=det_cfg.get('percentile', 89.0),
        downscale_factor=det_cfg.get('downscale_factor', 2),
        droplet_radius_px=det_cfg.get('droplet_radius_px', 100),
        min_distance=det_cfg.get('min_distance', 40),
        corr_backend=det_cfg.get('corr_backend', 'fft'),
        cpu_downscale=det_cfg.get('cpu_downscale', True),
        peak_downscale=det_cfg.get('peak_downscale', 1),
    )

state = AppState(root_dir=Path(cfg.get('recording', {}).get('root_dir', 'experiments')))
print('Detector:', type(detector).__name__)


Detector: DropletDetectorCuPy


In [4]:
controls = make_basic_controls(default_root=cfg.get('recording', {}).get('root_dir', 'experiments'))

# Camera/source defaults
controls['fps'].value = cfg.get('camera', {}).get('fps', 20.0)
controls['exposure_ms'].value = cfg.get('camera', {}).get('exposure_ms', 50.0)
controls['source_kind'].value = cfg.get('source', {}).get('kind', 'camera')
controls['video_path'].value = cfg.get('source', {}).get('video_path', '')
controls['video_loop'].value = cfg.get('source', {}).get('video_loop', True)
controls['fake_realtime'].value = cfg.get('source', {}).get('fake_realtime', True)

# Detector defaults
controls['percentile'].value = det_cfg.get('percentile', 89.0)
controls['downscale_factor'].value = det_cfg.get('downscale_factor', 2)
controls['droplet_radius_px'].value = det_cfg.get('droplet_radius_px', 100)
controls['min_distance'].value = det_cfg.get('min_distance', 40)
controls['corr_backend'].value = det_cfg.get('corr_backend', 'fft')
controls['cpu_downscale'].value = det_cfg.get('cpu_downscale', True)
controls['peak_downscale'].value = det_cfg.get('peak_downscale', 1)

# Recording defaults
rec_cfg = cfg.get('recording', {})
controls['duration_s'].value = rec_cfg.get('duration_s', 60.0)
controls['roi_radius_px'].value = rec_cfg.get('roi_radius_px', 80)
controls['output_name'].value = rec_cfg.get('output_name', 'droplet_intensity.tsv')
controls['raw_video_name'].value = rec_cfg.get('raw_video_name', 'raw_video.avi')
controls['overlay_video_name'].value = rec_cfg.get('overlay_video_name', 'overlay_video.avi')

# Preview defaults
prev_cfg = cfg.get('preview', {})
controls['colorize'].value = prev_cfg.get('colorize', True)
controls['draw_ids'].value = prev_cfg.get('draw_ids', True)
controls['draw_detections'].value = prev_cfg.get('draw_detections', False)
controls['beep'].value = prev_cfg.get('beep_on_crossing', True)
controls['reset_tracker'].value = prev_cfg.get('reset_tracker_on_start', True)

display_controls(controls)
_handle = bind_basic_callbacks(controls, state, detector)


## Notes

- `Start preview/stream` opens an OpenCV preview window. It works with either the live Basler camera or a prerecorded video file.
- Detector parameters can be changed during setup and applied without editing code.
- `Save raw frame` and `Save overlay frame` save the last acquired frame into the experiment root directory.
- Enable `Record raw video` or `Record overlay video` before starting preview to save video while streaming.
- `Record intensities` writes a TSV table where each selected droplet ID is a separate intensity channel.
- If the `IDs` field is empty, the currently active tracker IDs are used.
- Use `Favorite ID` plus `Finish zone` to highlight a droplet and trigger the collection signal when it crosses the zone.
